In [2]:
print("Hello World")

Hello World


In [49]:
# Loading in a dataset
import seaborn as sns
titanic_df = sns.load_dataset("titanic")
titanic_df.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [50]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder

In [51]:
display(titanic_df)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


Part 1

In [52]:
titanic_df.isnull().values.any

<function ndarray.any>

In [53]:
missing_vals = titanic_df.isnull().sum()
print(missing_vals)

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


Part 2

In [54]:
titanic_df.duplicated().sum()

107

In [55]:
dups = titanic_df[titanic_df.duplicated()]
print(dups)

     survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
47          1       3  female   NaN      0      0   7.7500        Q   Third   
76          0       3    male   NaN      0      0   7.8958        S   Third   
77          0       3    male   NaN      0      0   8.0500        S   Third   
87          0       3    male   NaN      0      0   8.0500        S   Third   
95          0       3    male   NaN      0      0   8.0500        S   Third   
..        ...     ...     ...   ...    ...    ...      ...      ...     ...   
870         0       3    male  26.0      0      0   7.8958        S   Third   
877         0       3    male  19.0      0      0   7.8958        S   Third   
878         0       3    male   NaN      0      0   7.8958        S   Third   
884         0       3    male  25.0      0      0   7.0500        S   Third   
886         0       2    male  27.0      0      0  13.0000        S  Second   

       who  adult_male deck  embark_town alive  alo

In [56]:
titanic_df = titanic_df[~titanic_df.isin(dups).all(axis=1)]
print(titanic_df)

     survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
0           0       3    male  22.0      1      0   7.2500        S   Third   
1           1       1  female  38.0      1      0  71.2833        C   First   
2           1       3  female  26.0      0      0   7.9250        S   Third   
3           1       1  female  35.0      1      0  53.1000        S   First   
4           0       3    male  35.0      0      0   8.0500        S   Third   
..        ...     ...     ...   ...    ...    ...      ...      ...     ...   
886         0       2    male  27.0      0      0  13.0000        S  Second   
887         1       1  female  19.0      0      0  30.0000        S   First   
888         0       3  female   NaN      1      2  23.4500        S   Third   
889         1       1    male  26.0      0      0  30.0000        C   First   
890         0       3    male  32.0      0      0   7.7500        Q   Third   

       who  adult_male deck  embark_town alive  alo

Part 3

In [57]:
titanic_df["sex"].unique()

array(['male', 'female'], dtype=object)

In [58]:
titanic_df["sex"].value_counts(dropna=False)

sex
male      577
female    313
Name: count, dtype: int64

Part 4

In [61]:
import pandas as pd

# --- 1. Function to calculate IQR bounds ---
def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

In [81]:
Q1_age = titanic_df['age'].quantile(0.25)
Q3_age = titanic_df['age'].quantile(0.75)
IQR_age = Q3_age - Q1_age

Q1_fare = titanic_df['fare'].quantile(0.25)
Q3_fare = titanic_df['fare'].quantile(0.75)
IQR_fare = Q3_fare - Q1_fare
print(IQR_age)
print(IQR_fare)

18.0
23.096899999999998


In [84]:
# Detect outliers in 'Age' and 'Fare' columns
age_outliers = titanic_df[(titanic_df['age'] < (Q1_age - 1.5 * IQR_age)) | (titanic_df['age'] > (Q3_age + 1.5 * IQR_age))]
fare_outliers = titanic_df[(titanic_df['fare'] < (Q1_fare - 1.5 * IQR_fare)) | (titanic_df['fare'] > (Q3_fare + 1.5 * IQR_fare))]

In [85]:

# Print outliers
print("\nAge outliers:")
print(age_outliers[['age', 'fare']])


Age outliers:
      age     fare
33   66.0  10.5000
96   71.0  34.6542
116  70.5   7.7500
493  71.0  49.5042
630  80.0  30.0000
672  70.0  10.5000
745  70.0  71.0000
851  74.0   7.7750


In [86]:

print("\nFare outliers:")
print(fare_outliers[['age', 'fare']])


Fare outliers:
      age      fare
1    38.0   71.2833
27   19.0  263.0000
31    NaN  146.5208
34   28.0   82.1708
52   49.0   76.7292
..    ...       ...
846   NaN   69.5500
849   NaN   89.1042
856  45.0  164.8667
863   NaN   69.5500
879  56.0   83.1583

[115 rows x 2 columns]


In [87]:

# Remove outliers (optional)
titanic_df = titanic_df[~((titanic_df['age'] < (Q1_age - 1.5 * IQR_age)) | (titanic_df['age'] > (Q3_age + 1.5 * IQR_age)))]
titanic_df = titanic_df[~((titanic_df['fare'] < (Q1_fare - 1.5 * IQR_fare)) | (titanic_df['fare'] > (Q3_fare + 1.5 * IQR_fare)))]

In [88]:

# Verify changes
print(f"\nRows after removing outliers: {titanic_df.shape[0]}")


Rows after removing outliers: 768


Part 5

Tasks later in the day (did not get to do 5-9)


In [96]:
display(titanic_df)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


Afternoon Tasks

In [109]:
# Loading in a dataset
import seaborn as sns
titanic_df = sns.load_dataset("titanic")
titanic_df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


Task 1
create a function to take the titanic_df and return with no dups or na's

In [110]:
def clean_titanic(df):
    """
    Return a cleaned version of the Titanic dataframe
    with no duplicates and no missing values.
    """
    return df.drop_duplicates().dropna().reset_index(drop=True)

In [111]:
clean_df = clean_titanic(titanic_df)

In [99]:
display(clean_df)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
1,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
2,1,3,female,4.0,1,1,16.7000,S,Third,child,False,G,Southampton,yes,False
3,1,1,female,58.0,0,0,26.5500,S,First,woman,False,C,Southampton,yes,True
4,1,2,male,34.0,0,0,13.0000,S,Second,man,True,D,Southampton,yes,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,0,1,male,31.0,0,0,50.4958,S,First,man,True,A,Southampton,no,True
91,1,1,female,47.0,1,1,52.5542,S,First,woman,False,D,Southampton,yes,False
92,0,1,male,33.0,0,0,5.0000,S,First,man,True,B,Southampton,no,True
93,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True


Task 2
Numeric columns (6)
survived, pclass, age, sibsp, parch, fare

remove outliers and return two df's, one is a clean titanic one and the second is an outliers df

In [112]:
import pandas as pd

def remove_outliers(df):
    """
    Removes outliers from every numeric column using the IQR method.
    Returns:
        clean_df: dataframe without outliers
        outliers_df: dataframe containing only outlier rows
    """
    numeric_cols = df.select_dtypes(include='number').columns

    # Compute IQR for each numeric column
    Q1 = df[numeric_cols].quantile(0.25)
    Q3 = df[numeric_cols].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Identify outlier rows
    outlier_mask = (df[numeric_cols] < lower) | (df[numeric_cols] > upper)
    outlier_rows = outlier_mask.any(axis=1)

    # Split into two DataFrames
    outliers_df = df[outlier_rows].reset_index(drop=True)
    clean_df = df[~outlier_rows].reset_index(drop=True)

    return clean_df, outliers_df

In [113]:
display(clean_df)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
1,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
2,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
3,1,3,female,4.0,1,1,16.7000,S,Third,child,False,G,Southampton,yes,False
4,1,1,female,58.0,0,0,26.5500,S,First,woman,False,C,Southampton,yes,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,1,1,female,47.0,1,1,52.5542,S,First,woman,False,D,Southampton,yes,False
177,0,1,male,33.0,0,0,5.0000,S,First,man,True,B,Southampton,no,True
178,1,1,female,56.0,0,1,83.1583,C,First,woman,False,C,Cherbourg,yes,False
179,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True


In [116]:
clean_df, outliers_df = remove_outliers(titanic_df)

In [117]:
display(outliers_df)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
1,0,3,male,2.0,3,1,21.0750,S,Third,child,False,NaN,Southampton,no,False
2,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
3,1,3,female,4.0,1,1,16.7000,S,Third,child,False,G,Southampton,yes,False
4,0,3,male,39.0,1,5,31.2750,S,Third,man,True,NaN,Southampton,no,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288,1,1,female,47.0,1,1,52.5542,S,First,woman,False,D,Southampton,yes,False
289,1,1,female,56.0,0,1,83.1583,C,First,woman,False,C,Cherbourg,yes,False
290,1,2,female,25.0,0,1,26.0000,S,Second,woman,False,NaN,Southampton,yes,False
291,0,3,female,39.0,0,5,29.1250,Q,Third,woman,False,NaN,Queenstown,no,False
